In [3]:
!pip install pyspark -q

In [7]:
from pyspark.sql import SparkSession

#inicjalizacja sesji
spark = SparkSession.builder \
    .appName("Sales") \
    .master("local[*]") \
    .getOrCreate()

#wczytanie pliku
df = spark.read.csv("dane.csv", header=True, inferSchema=True)

#pokazanie danych
df.show(5)

+--------+-----------+----+-----+
| Produkt|  Kategoria|Cena|Ilosc|
+--------+-----------+----+-----+
|   Chleb|  Spozywcze|   4|  100|
|   Mleko|  Spozywcze|   4|  150|
| Telefon|Elektronika|3500|    3|
|Koszulka|     Odziez|  55|   50|
|  Laptop|Elektronika|5000|   10|
+--------+-----------+----+-----+
only showing top 5 rows


In [5]:
from pyspark.sql import functions as F

#podstawowe operacje
print("\nWyświetlanie i schemat")
df.show()
df.printSchema()

print("Selekcja kolumn")
df.select("Produkt", "Cena").show()

print("Filtrowanie wierszy")
df.filter(df["Cena"] > 100).show()

print("Grupowanie i agregacje")
#dodatkowa kolumna z wartością (cena x ilosc)
df_value = df.withColumn("Wartosc", df["Cena"] * df["Ilosc"])
df_agr = df_value.groupBy("Kategoria").agg(F.sum("Wartosc").alias("Suma_Sprzedazy"))
df_agr.show()


#zapis
wyjsciowy_folder_parquet = "wynik_parquet"

df_agr.write.mode("overwrite").parquet(wyjsciowy_folder_parquet)

spark.stop()


Wyświetlanie i schemat
+--------+-----------+----+-----+
| Produkt|  Kategoria|Cena|Ilosc|
+--------+-----------+----+-----+
|   Chleb|  Spozywcze|   4|  100|
|   Mleko|  Spozywcze|   4|  150|
| Telefon|Elektronika|3500|    3|
|Koszulka|     Odziez|  55|   50|
|  Laptop|Elektronika|5000|   10|
|    Buty|     Odziez| 150|   80|
|Koszulka|     Odziez|  60|   50|
|    Kawa|  Spozywcze|  25|   35|
+--------+-----------+----+-----+

root
 |-- Produkt: string (nullable = true)
 |-- Kategoria: string (nullable = true)
 |-- Cena: integer (nullable = true)
 |-- Ilosc: integer (nullable = true)

Selekcja kolumn
+--------+----+
| Produkt|Cena|
+--------+----+
|   Chleb|   4|
|   Mleko|   4|
| Telefon|3500|
|Koszulka|  55|
|  Laptop|5000|
|    Buty| 150|
|Koszulka|  60|
|    Kawa|  25|
+--------+----+

Filtrowanie wierszy
+-------+-----------+----+-----+
|Produkt|  Kategoria|Cena|Ilosc|
+-------+-----------+----+-----+
|Telefon|Elektronika|3500|    3|
| Laptop|Elektronika|5000|   10|
|   Buty|   

In [6]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Analiza RDD") \
    .master("local[*]") \
    .getOrCreate()
sc = spark.sparkContext

#wczytanie pliku jako surowy plik tekstowy RDD
surowe_rdd = sc.textFile("dane.csv")

#odfiltrowanie nagłówka
naglowek = surowe_rdd.first()
dane_bez_naglowka = surowe_rdd.filter(lambda linia: linia != naglowek)

#parsowanie
parsuj_rdd = dane_bez_naglowka.map(lambda linia: linia.split(",")) \
                              .map(lambda kolumny: [kolumny[0], kolumny[1], float(kolumny[2]), int(kolumny[3])])

#transformacje
drogie_produkty_rdd = parsuj_rdd.filter(lambda x: x[2] > 100)
print("\nProdukty z ceną > 100:")
print(drogie_produkty_rdd.collect())

liczba_wierszy = parsuj_rdd.count()
print(f"\nLiczba wierszy w pliku: {liczba_wierszy}")

sama_ilosc_rdd = parsuj_rdd.map(lambda x: x[3])
calkowita_ilosc = sama_ilosc_rdd.reduce(lambda a, b: a + b)
print(f"Ilość wszystkich produktów: {calkowita_ilosc}")



Produkty z ceną > 100:
[['Telefon', 'Elektronika', 3500.0, 3], ['Laptop', 'Elektronika', 5000.0, 10], ['Buty', 'Odziez', 150.0, 80]]

Liczba wierszy w pliku: 8
Ilość wszystkich produktów: 478
